Cell1: Complete Data Loading and Cleaning Pipeline

In [ ]:
import pandas as pd
import json

def process_job_data(file_path):
    """
    Comprehensive data loading and cleaning pipeline for 1M+ records.
    Ensures all columns are retained and optimized for downstream insights.
    """
    # 1. Complete optimal data types to minimize memory across all features
    optimized_dtypes = {
        'employmentTypes': 'category',
        'metadata_isPostedOnBehalf': 'boolean',
        'metadata_jobPostId': 'string',
        'metadata_repostCount': 'Int16',
        'metadata_totalNumberJobApplication': 'Int32', 
        'metadata_totalNumberOfView': 'Int32',
        'minimumYearsExperience': 'Int8',
        'numberOfVacancies': 'Int16',
        'positionLevels': 'category',
        'postedCompany_name': 'string',
        'salary_maximum': 'float32',
        'salary_minimum': 'float32',
        'salary_type': 'category',
        'status_jobStatus': 'category',
        'title': 'string',
        'average_salary': 'float32'
    }
    
    # 2. Define date columns for automatic parsing (vital for time-series insights)
    date_cols = [
        'metadata_expiryDate', 
        'metadata_newPostingDate', 
        'metadata_originalPostingDate'
    ]

    print("Loading data...")
    # 3. Load the CSV file
    df = pd.read_csv(
        file_path,
        dtype=optimized_dtypes,
        parse_dates=date_cols,
        dayfirst=True,  # Ensures dates like 22/4/2023 are parsed correctly
        low_memory=False,
        nrows = 50000
    )
    
    # 4. Drop columns that are completely empty
    if 'occupationId' in df.columns:
        df = df.drop(columns=['occupationId'])
        print("Dropped empty 'occupationId' column.")

    # 5. Parse the JSON string in 'categories'
    print("Parsing JSON categories...")
    def extract_categories(json_str):
        if pd.isna(json_str):
            return []
        try:
            parsed = json.loads(json_str)
            return [item.get('category') for item in parsed if 'category' in item]
        except (json.JSONDecodeError, TypeError):
            return []

    # Apply extraction and drop the original raw JSON string column
    df['parsed_categories'] = df['categories'].apply(extract_categories)
    df = df.drop(columns=['categories'])
    
    print("Pipeline execution complete. DataFrame is ready for analysis.")
    return df

# Execute the function and store it in memory for all subsequent cells
# Replace 'jobs_dataset.csv' with your actual file
df_clean = process_job_data('SGJobData.csv')

# Verify the fully loaded and cleaned dataframe
df_clean.info()


In [ ]:
#Run this code to install tabulate
%conda install -c conda-forge tabulate -y

Cell2: Mismatch cell analysis

In [ ]:
import numpy as np

# 1. Explode the pre-parsed categories so each job can be counted under multiple sectors if necessary
df_exploded = df_clean.explode('parsed_categories')

# 2. Aggregate Market Signals by Sector
print("Calculating supply and demand metrics...")
sector_metrics = df_exploded.groupby('parsed_categories').agg(
    total_postings=('metadata_jobPostId', 'count'),
    total_vacancies=('numberOfVacancies', 'sum'),
    total_applications=('metadata_totalNumberJobApplication', 'sum'),
    avg_repost_count=('metadata_repostCount', 'mean'),
    avg_salary=('average_salary', 'mean'),
    avg_years_exp=('minimumYearsExperience', 'mean')
).reset_index()

# 3. Calculate Critical Efficiency Ratios
# Applications per Vacancy (Supply Density)
sector_metrics['apps_per_vacancy'] = (
    sector_metrics['total_applications'] / sector_metrics['total_vacancies']
).replace([np.inf, -np.inf], np.nan)

# 4. Identify Shortages (High Reposting, Low Supply)
# Sorting by highest repost count and lowest applications per vacancy
shortages = sector_metrics.sort_values(
    by=['avg_repost_count', 'apps_per_vacancy'], 
    ascending=[False, True]
).dropna(subset=['apps_per_vacancy'])

# 5. Display Results
columns_to_display = [
    'parsed_categories', 'total_vacancies', 'avg_repost_count', 
    'apps_per_vacancy', 'avg_salary'
]
print("\n--- Top 10 Sectors Experiencing Critical Skills Gaps ---")
print(shortages[columns_to_display].head(10).to_markdown(index=False))
